# Week 3, Notebook 2: Monte Carlo Simulation

**Rolling the dice on your future to see the odds.**

*Author: The Genius Project Year 3*

---

### What you will build
- A simulator that plays out one uncertain year of saving.
- Then plays it out **ten thousand times** to see every way it could go.
- A real probability: *the chance you reach your savings goal.*

### The big idea, in one breath
You cannot know the future, but you know roughly how your months behave. So you let
the computer invent thousands of believable futures, each a little different, and
count how many end well. That trick &mdash; answering a hard question with a huge pile
of random tries &mdash; is called **Monte Carlo simulation**, named after the famous
casino.

### Step 1: Learn how a typical month behaves

From Notebook 1 we know saving is uncertain: some months are great, some are
negative. Monte Carlo needs two numbers to describe that uncertainty &mdash; the
**average** amount saved and the **spread** (standard deviation) around it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

budget = pd.read_csv("data/monthly_budget.csv")

avg_saved = budget["saved"].mean()
spread_saved = budget["saved"].std()

print(f"On an average month you save:  ${avg_saved:.2f}")
print(f"But it swings up and down by about ${spread_saved:.2f} (the spread).")

### Step 2: Play out ONE random year

A future month is a random draw: centred on your average, wobbling by your spread.
We draw 12 of them, add them to a fresh empty jar, and see where we land. Run this
box a few times &mdash; you get a different year every time. That is the whole point.

In [ ]:
rng = np.random.default_rng()  # the source of randomness

one_year = rng.normal(avg_saved, spread_saved, size=12)  # 12 random months
jar = one_year.cumsum()  # the running total through the year

print("Money saved each month:", np.round(one_year, 0))
print(f"Total in the jar after 12 months: ${jar[-1]:.2f}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 13), jar, marker="o")
plt.title("One possible year of saving")
plt.xlabel("Month")
plt.ylabel("Money in the jar ($)")
plt.grid(alpha=0.3)
plt.show()

### Step 3: Now play it out 10,000 times

One year tells you almost nothing &mdash; it might have been lucky or unlucky. The
Monte Carlo move is to simulate **thousands** of years and look at the whole crowd
of outcomes. We use a fixed seed so your results match this notebook.

In [ ]:
rng = np.random.default_rng(42)   # fixed seed = reproducible results
N = 10_000                        # number of imagined futures
MONTHS = 12

# Each row is one imagined year of 12 random months; sum each row to get its total.
all_futures = rng.normal(avg_saved, spread_saved, size=(N, MONTHS))
final_totals = all_futures.sum(axis=1)

print(f"Simulated {N:,} different years.")
print(f"Typical year ends with about ${np.median(final_totals):.0f} saved.")
print(f"Best 1% of years reach ${np.percentile(final_totals, 99):.0f} or more.")
print(f"Worst 1% of years end below ${np.percentile(final_totals, 1):.0f}.")

### Step 4: What are the odds of hitting a goal?

Say you want **$500** for a new phone by the end of the year. The probability is
simply the share of your 10,000 imagined years that reached it. No formula needed
&mdash; you just count.

In [ ]:
GOAL = 500

reached = final_totals >= GOAL
chance = reached.mean()  # True counts as 1, so the mean is the fraction that won
print(f"Chance of reaching ${GOAL} in a year: {chance * 100:.1f}%")

plt.figure(figsize=(9, 4))
plt.hist(final_totals, bins=40, color="#4C72B0", edgecolor="white")
plt.axvline(GOAL, color="#C44E52", linestyle="--", linewidth=2, label=f"goal ${GOAL}")
plt.title(f"10,000 imagined years  (you hit the goal {chance*100:.0f}% of the time)")
plt.xlabel("Total saved after 12 months ($)")
plt.ylabel("Number of imagined years")
plt.legend()
plt.show()

### Step 5: Change one habit, change the odds

Here is where simulation earns its keep. Suppose you drop one subscription and save
**$15 more every month**. You do not need to live the year to see the effect &mdash;
just shift the average and re-run the 10,000 futures.

In [ ]:
extra = 15  # save $15 more each month
better_futures = rng.normal(avg_saved + extra, spread_saved, size=(N, MONTHS))
better_totals = better_futures.sum(axis=1)
better_chance = (better_totals >= GOAL).mean()

print(f"Before: {chance * 100:.1f}% chance of reaching ${GOAL}")
print(f"After saving $15 more a month: {better_chance * 100:.1f}% chance")
print(f"That one habit changed your odds by {(better_chance - chance) * 100:.0f} points.")

plt.figure(figsize=(9, 4))
plt.hist(final_totals, bins=40, alpha=0.6, label="as you are now", color="#4C72B0")
plt.hist(better_totals, bins=40, alpha=0.6, label="saving $15 more", color="#55A868")
plt.axvline(GOAL, color="#C44E52", linestyle="--", linewidth=2, label=f"goal ${GOAL}")
plt.title("One small habit shifts the whole future")
plt.xlabel("Total saved after 12 months ($)")
plt.ylabel("Number of imagined years")
plt.legend()
plt.show()

### What you learned
- **Monte Carlo** answers "what are my chances?" by trying thousands of random
  futures and counting the good ones.
- A probability can be an honest **count**, not a scary formula.
- **Percentiles** describe the good and bad cases, not just the average.
- Simulation lets you test a change (*save $15 more*) before you live it.

### Try it yourself
- Raise `GOAL` to `$800`. How much does the chance drop?
- Change `MONTHS` to `24` (a two-year plan). What happens to the odds?
- Banks call the worst-5% outcome the **Value at Risk**. Print
  `np.percentile(final_totals, 5)` &mdash; that is the number a cautious planner watches.

### Next
So far the months were independent draws. Real money and real prices move in
**time**, where today depends on yesterday. Notebook 3 opens the world of
**time series** and the most important idea in it: **stationarity**.